# MCMC-PESP — Przewodnik użytkownika

Kompletna instrukcja obsługi projektu: od nowej sieci do pełnej analizy.

## Spis treści
1. [Struktura projektu](#1-struktura-projektu)
2. [Kluczowe abstrakcje i API](#2-kluczowe-abstrakcje-i-api)
3. [Dobór parametrów — reguły kciuka](#3-dobór-parametrów)
4. [Workflow: nowa sieć → analiza wstępna](#4-workflow-nowa-sieć)
5. [Parallel Tempering — framework kalibracji](#5-parallel-tempering)
6. [Diagnostyki zbieżności](#6-diagnostyki-zbieżności)
7. [Porównanie propozycji ruchów](#7-porównanie-propozycji)
8. [Kalibracja funkcji kosztu](#8-kalibracja-funkcji-kosztu)
9. [Benchmark metod](#9-benchmark-metod)
10. [Skalowanie i profilowanie](#10-skalowanie-i-profilowanie)
11. [Zapis i wczytywanie wyników](#11-zapis-wyników)

---
**Konwencja notebooka:**  
- Komórki `# CONFIG` — tu zmieniasz parametry  
- Komórki `# RUN` — uruchamiają obliczenia  
- Komórki `# ANALYSIS` — interpretacja wyników

---
## 1. Struktura projektu

```
mcmc-pesp/
├── src/
│   ├── core/              Struktury danych (Network, Line, Schedule, Transfer)
│   ├── cost/              Funkcje kosztu (TransferCost, OperationalCost, TotalCost)
│   ├── mcmc/
│   │   ├── sampler.py     Główna pętla MH (MCMCSampler)
│   │   ├── parallel_tempering.py  PT z inercją
│   │   ├── adaptation.py  ProposalAdapter (Mode A/B)
│   │   └── proposals/     SingleLineShift, ComponentMove, SmartRepair
│   ├── io/
│   │   ├── synthetic.py          Sieć trójkąta (PoC)
│   │   └── synthetic_scenarios.py  7 typów scenariuszy syntetycznych
│   ├── metrics/           R-hat, ESS, peak_concurrency
│   └── experiments/
│       ├── analysis/
│       │   ├── diagnostic_runner.py   Wszystkie funkcje diagnostyczne
│       │   ├── pt_stress_test.py      Stress test PT
│       │   └── pt_calibration.py      Auto-kalibracja beta dla PT
│       └── benchmark/
│           └── run_synthetic_benchmark.py
├── notebooks/
│   ├── 01_poc_triangle.ipynb      PoC na sieci trójkąta
│   ├── 04_diagnostic_analysis.ipynb  Pełna diagnostyka
│   ├── 05_user_guide.ipynb        Ten plik
│   ├── 06_benchmark_comparison.ipynb
│   └── 08_profiling_optimization.ipynb
├── configs/
│   ├── default.yaml
│   ├── poc.yaml
│   ├── benchmark.yaml
│   └── synthetic_benchmark.yaml
├── results/diagnostics/   Wyniki diagnostyk
└── docs/MATH_SPEC.md      Pełna specyfikacja matematyczna
```

In [ ]:
# SETUP — uruchom raz na początku każdej sesji
import sys, os, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['font.size'] = 11

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Core imports
from src.core.network import Network, Line, Station, T_PERIOD
from src.core.schedule import Schedule
from src.core.transfers import Transfer, TransferParams, TransferSet

# Cost
from src.cost.transfer_cost import TransferCost
from src.cost.operational import OperationalCost, TurnaroundCost, FleetCost
from src.cost.robustness import RobustnessCost, DelayDistribution
from src.cost.total import TotalCost

# MCMC
from src.mcmc.sampler import MCMCSampler
from src.mcmc.parallel_tempering import ParallelTempering, InertiaConfig
from src.mcmc.proposals.single_line import SingleLineShift, UniformLineShift
from src.mcmc.proposals.component import create_component_move
from src.mcmc.proposals.smart_repair import create_smart_repair

# Scenarios & diagnostics
from src.io.synthetic_scenarios import SyntheticScenarioFactory
from src.experiments.analysis.diagnostic_runner import (
    sample_cost_landscape, run_convergence_diagnostic,
    run_proposal_comparison, run_pt_diagnostic,
    run_scaling_analysis, run_optimum_gap_tracking,
    run_barrier_sensitivity, run_lambda_sweep, compute_pareto_front,
    run_random_search, run_simulated_annealing, run_benchmark_comparison,
    run_profiling_analysis,
    build_cost_fn, build_proposals,
)
from src.metrics.convergence import compute_r_hat, compute_ess, MCMCDiagnostics
from src.experiments.analysis.pt_calibration import (
    estimate_cost_stats, calibrate_beta_schedule, run_pt_with_betas,
)

factory = SyntheticScenarioFactory()
print(f'Project root: {PROJECT_ROOT}')
print(f'T_PERIOD = {T_PERIOD} min')
print(f'Available scenario types: {factory.list_scenarios()}')

---
## 2. Kluczowe abstrakcje i API

### 2.1 Scenariusze syntetyczne — SyntheticScenarioFactory

| Typ | Opis | Difficulty (linie) | Optymum znane? |
|-----|------|--------------------|----------------|
| `hub` | Gwiazda: N/2 przyjeżdżających × N/2 odjeżdżających | easy=6, medium=8, hard=12 | TAK |
| `chain` | Łańcuch liniowy L₀→L₁→…→L_n | easy=3, medium=5, hard=8 | TAK |
| `conflicting` | Pierścień z niespójnymi wymaganiami | easy=3, medium=4, hard=5 | NIE (brak opt. 0) |
| `weighted` | Hub z niejednorodną wagą przesiadek | jak hub | NIE |
| `full_cost` | Hub + obroty taboru (turnaround) | easy=4L, medium=8, hard=12 | NIE |
| `multi_hub_mesh` | 3 węzły, IC/Regional/Local, mieszane freq | easy=12L, medium=18, hard=18 | NIE |
| `random_network` | Losowa topologia | easy=6L/9T, medium=12L/38T, hard=20L/82T | NIE |

**Wspólne `**kwargs`** (wszystkie typy):
```python
frequency=60        # częstotliwość [min]
travel_time=15      # czas jazdy [min]
dwell_time=2        # czas postoju [min]
walk_time=3.0       # czas przejścia na przesiadce [min]
w_min=3.0           # minimalne oczekiwanie [min]
w_ideal=8.0         # idealne oczekiwanie [min]
w_max=30.0          # maksymalne oczekiwanie [min]
alpha=2.0           # kara poniżej w_ideal
beta=0.5            # kara powyżej w_ideal
M=1e6               # bariera kwadratowa
```

### 2.2 MCMCSampler — parametry

```python
MCMCSampler(
    network,
    cost_fn,
    proposals,            # lista propozycji
    proposal_weights,     # lista wag (suma=1, domyślnie równe)
    beta=1.0,             # odwrotna temperatura
    mode='optimization',  # 'optimization' (best-so-far) | 'sampling' (formalny MCMC)
    rng,
)
sampler.initialize(Schedule.random(network, rng))  # lub konkretny rozkład
result = sampler.run(n_steps=50000, burn_in=5000, thin=10, log_every=5000)
```

### 2.3 ParallelTempering — parametry

```python
pt = ParallelTempering(
    network, cost_fn, proposals, proposal_weights,
    n_replicas=8,
    beta_min=0.5,    # najgorętsza replika (idx=0)
    beta_max=3.0,    # najzimniejsza replika (idx=n-1)
    swap_interval=10,
    mode='optimization',
    inertia_config=None,   # lub InertiaConfig(modality_inertia={...})
    rng,
)
pt.initialize()   # losowe startowe rozkłady jazdy
result = pt.run(n_steps=steps_per_replica, burn_in=0, thin=1)
```

### 2.4 Propozycje ruchów — sygnatury

```python
SingleLineShift(sigma=5.0, rng=rng)            # Gauss, 1 linia
UniformLineShift(max_shift=10, rng=rng)        # Uniform, 1 linia
create_component_move(transfers, sigma=5.0, p_decay=None, rng=rng)  # BFS-spójny
create_smart_repair(transfers, sigma=2.0, rng=rng)                  # celowana naprawa
```

### 2.5 Funkcje kosztu

```python
# Tylko PAX (przesiadki) — najczęściej używana w diagnostykach
cost_fn = TransferCost(transfers)

# Pełna (PAX + OPS + ROB)
cost_fn = TotalCost(
    transfer_cost=TransferCost(transfers),
    operational_cost=OperationalCost(TurnaroundCost(turnarounds)),  # opcjonalnie
    robustness_cost=RobustnessCost(transfers, DelayDistribution(rate=1/3, alpha=0.95)),
    pax_weight=1.0, ops_weight=0.1, rob_weight=0.05,
)
```

---
## 3. Dobór parametrów — reguły kciuka

### 3.1 Beta (odwrotna temperatura)

Beta kontroluje stosunek eksploracji do eksploatacji:
- **Beta za małe** (→ 0): łańcuch akceptuje wszystko — błądzi losowo, nie optymalizuje
- **Beta za duże** (→ ∞): łańcuch akceptuje tylko ruchy w dół — hill climbing, utyka w lokalnych minimach
- **Beta optymalne**: acceptance rate ≈ 20–40%

**Procedura wyboru beta:**
```
1. Uruchom krótki test (1000 kroków) dla kilku wartości beta: [0.01, 0.1, 0.5, 1.0, 3.0]
2. Sprawdź acceptance rate → wybierz beta dające 20–40%
3. Jeśli acceptance ≈ stały dla beta=0.5 i beta=3.0 → łańcuch jest 'zamrożony'
   (koszty są zbyt wysokie dla tych beta — patrz sekcja 5 o PT)
```

| Sieć (z doświadczeń) | Zalecane beta | Typowy acceptance |
|---|---|---|
| hub_easy | 1.0–2.0 | 37% |
| hub_hard | 1.0–2.0 | 38% |
| chain | 1.0–2.0 | 22% |
| multi_hub_mesh | 1.0–3.0 | 25–28% |
| random_network_hard | **0.001–0.01** | 37–38% |

### 3.2 Propozycje i ich wagi

**Sprawdzone wagi domyślne** (z diagnostic_runner.py):
```python
weights = [0.4, 0.1, 0.3, 0.2]  # single, uniform, component, smart_repair
```

**Kiedy modyfikować:**

| Sytuacja | Zmiana |
|---|---|
| Dużo naruszeń przesiadek | ↑ smart_repair (do 0.3–0.4) |
| Mało przesiadek (<5) | wyłącz component_move i smart_repair |
| Sieć bardzo liniowa | ↑ component_move (przesuwa całą trasę) |
| Chcesz max ESS/step | użyj tylko smart_repair (ESS/step 6× wyższe) |

**Wyniki porównania propozycji na multi_hub_mesh_easy:**

| Propozycja | Accept. | Best cost | ESS/step |
|---|---|---|---|
| single_line_shift | 17.5% | 3200 | 0.0061 |
| uniform_line_shift | 14.2% | **3114** | 0.0071 |
| component_move | **53.1%** | **3114** | 0.0019 |
| **smart_repair** | 7.4% | 3184 | **0.0386** |
| **full_mix** | 26.3% | **3114** | 0.0104 |

### 3.3 Sigma propozycji

Sigma to std przesunięcia (w minutach) dla SingleLineShift i ComponentMove.

```
sigma zbyt małe (< 1): acceptance wysoki, ale małe ruchy → wolna eksploracja
sigma zbyt duże (> 30): acceptance spada drastycznie → mało akceptacji
sigma optymalne: zazwyczaj 3–10 min dla frequency=60 min
```

**Reguła:** `sigma ≈ frequency * 0.05 – 0.15` (5–15% okresu linii)

### 3.4 Parametry kosztu

| Parametr | Domyślnie | Kiedy zmieniać |
|---|---|---|
| `M` (bariera) | **1e5** | Nie zmieniaj powyżej 1e5 — wyniki identyczne od M≥1e4. Zmniejsz do 1e4 tylko jeśli chain nie opuszcza stanu startowego. |
| `pax_weight` | 1.0 | Stałe — skala referencyjna |
| `ops_weight` | 0.1 | Zwiększ jeśli obroty taboru są krytyczne; bez efektu gdy brak turnaroundów |
| `rob_weight` | **0.05** | **Nigdy > 0.1** — lambda_rob=1.0 pogarsza PAX 7× bez proporcjonalnego zysku ROB |
| `alpha` (kara < w_ideal) | 2.0 | ↑ alpha = bardziej penalizuj za krótki czas przesiadki |
| `beta_cost` (kara > w_ideal) | 0.5 | Typowo alpha >> beta (gorsze spóźnienie niż nadmiarowe czekanie) |
| `w_ideal` | 8.0 min | Ustaw na typowy czas przejścia + bufor |
| `delay_rate` | 1/3 (mean=3 min) | Ustaw na 1/mean_delay z danych historycznych |

**Skala bariery M a koszty:**  
> Jeśli koszty typowych losowych rozkładów są ~M (z landscape analysis), sieć jest zdominowana przez kary bariery.
> Sprawdź `cost_fn.count_violations(schedule, network)` na losowych próbkach.
> M = 1e6 i 36 przesiadek → max barrier = 36e6. Normalne koszty powinny być << M.

**Violation rate ≠ problem M:**
> Jeśli violation rate jest stały dla wszystkich wartości M (jak random_network_hard = 100%),
> problem leży w strukturze sieci — sprzeczne ograniczenia niemożliwe do jednoczesnego spełnienia.
> Miękka bariera działa poprawnie — minimalizuje naruszenia do minimum strukturalnego.

### 3.5 Liczba kroków

| Cel | Min. kroki | Rekomendowane |
|---|---|---|
| Szybki test / PoC | 5 000 | 10 000 |
| Diagnostyka zbieżności | 20 000 per chain | 50 000 |
| Optymalizacja małej sieci | 50 000 | 100 000 |
| Optymalizacja multi_hub_mesh | 100 000 | 500 000 |
| random_network_hard | 200 000 | 500 000 |
| PT (kroki per replika) | budget // n_replicas | jw. |

**Kiedy dość?** Gdy `best_cost` przestaje się poprawiać przez ostatnie 20% kroków.

---
## 4. Workflow: nowa sieć → analiza wstępna

Dla nowej sieci transportowej (niekoniecznie syntetycznej) zalecana kolejność:

```
Krok 1: Landscape analysis  → zrozum skalę kosztów i trudność
Krok 2: Probe betas         → znajdź zakres temperatur, gdzie chain się miesza
Krok 3: Krótka optymalizacja → sprawdź czy MCMC w ogóle działa
Krok 4: Zbieżność           → ile kroków potrzeba? Czy R-hat < 1.1?
Krok 5: Porównanie propozycji → który mix najlepszy?
Krok 6: PT (jeśli potrzeba) → kalibracja i uruchomienie
```

In [ ]:
# CONFIG — zmień na swoją sieć
SCENARIO_TYPE = 'multi_hub_mesh'   # lub 'random_network', 'hub', itp.
DIFFICULTY    = 'easy'             # 'easy', 'medium', 'hard'
SCENARIO_SEED = 42                 # dla random_network
SEED          = 42

# Utwórz scenariusz
kwargs = {'seed': SCENARIO_SEED} if SCENARIO_TYPE == 'random_network' else {}
scenario = factory.create(SCENARIO_TYPE, difficulty=DIFFICULTY, **kwargs)
print(f'Scenariusz: {scenario.metadata.name}')
print(f'  Linie: {scenario.metadata.n_lines}')
print(f'  Przesiadki: {scenario.metadata.n_transfers}')
print(f'  Obroty: {scenario.metadata.n_turnarounds}')
print(f'  Optymum znane: {scenario.metadata.optimal_solution is not None}')

In [ ]:
# KROK 1: Landscape analysis
# Losuje N_SAMPLES rozkładów jazdy i mierzy ich koszt — bez MCMC.
# Cel: zrozumieć skalę kosztów, rozrzut, stosunek min/median.

N_LANDSCAPE_SAMPLES = 500  # 500 = szybko; 2000 = dokładniej

landscape = sample_cost_landscape(scenario, n_samples=N_LANDSCAPE_SAMPLES, seed=SEED)

print(f'=== LANDSCAPE ANALYSIS: {landscape.scenario_name} ===')
print(f'  Min:    {landscape.min_cost:12.1f}  ← najlepszy losowy rozkład')
print(f'  Q5:     {landscape.q05:12.1f}')
print(f'  Median: {landscape.median:12.1f}')
print(f'  Mean:   {landscape.mean:12.1f}')
print(f'  Std:    {landscape.std:12.1f}')
print(f'  Q95:    {landscape.q95:12.1f}')
print(f'  Std/Mean: {landscape.std/landscape.mean:.2f}  (> 1 = trudny krajobraz)')
print(f'  Median/Min: {landscape.median/landscape.min_cost:.1f}×  (duże = optymum rzadkie)')

# Interpretacja:
# Std/Mean >> 1  → wiele lokalnych minimów, PT pomoże
# Median/Min >> 10 → znalezienie optimum wymaga kierowanej optymalizacji
# Min ≈ M (=1e6)  → problemem są bariery, nie koszty płynne

In [ ]:
# KROK 2: Probe betas — znajdź właściwy zakres temperatur
# Uruchamia krótkie łańcuchy dla kilku wartości beta.
# CEL: znaleźć beta_freeze — próg powyżej którego chain jest zamrożony.

PROBE_BETAS = [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 3.0]
N_WARMUP    = 1000
N_SAMPLE    = 2000

print(f'{'beta':>8}  {'mean':>12}  {'std':>12}  {'accept':>8}  Status')
print('-' * 60)
probe_results = []
prev_accept = None
for pb in PROBE_BETAS:
    s = estimate_cost_stats(scenario, pb, n_warmup=N_WARMUP, n_sample=N_SAMPLE, seed=SEED)
    # Zamrożenie: accept przestaje rosnąć gdy zmniejszamy beta
    frozen = (prev_accept is not None and abs(s.acceptance_rate - prev_accept) < 0.02)
    status = 'FROZEN (chain nie miesza!)' if frozen and s.acceptance_rate < 0.5 else 'ok'
    print(f'{pb:8.4f}  {s.mean:12.1f}  {s.std:12.1f}  {s.acceptance_rate:8.3f}  {status}')
    probe_results.append({'beta': pb, 'mean': s.mean, 'std': s.std, 'accept': s.acceptance_rate})
    prev_accept = s.acceptance_rate

# Jeśli acceptance ≈ stały dla wielu beta → zamrożenie!
# Dla random_network_hard: accept=0.39 dla beta=0.5, 1.0, 3.0 → zamrożony od beta=0.5
# Dla multi_hub_mesh: accept spada stopniowo → normalne
print()
print('Wnioski:')
accepts = [r['accept'] for r in probe_results]
if max(accepts) - min(accepts) < 0.1:
    print('  UWAGA: chain jest zamrożony we wszystkich badanych temperaturach!')
    print('  Użyj znacznie mniejszego beta lub znormalizuj koszty.')
else:
    best_beta = min([r['beta'] for r in probe_results if r['accept'] >= 0.20 and r['accept'] <= 0.45], 
                    default=probe_results[len(probe_results)//2]['beta'])
    print(f'  Zalecane beta dla optymalizacji: ~{best_beta}')

In [ ]:
# KROK 3: Krótka optymalizacja — sprawdź czy MCMC działa

# CONFIG
BETA          = 1.0      # ustaw na podstawie Kroku 2
N_STEPS_QUICK = 20000

rng = np.random.default_rng(SEED)
cost_fn = build_cost_fn(scenario)
proposals, weights = build_proposals(scenario, rng)

sampler = MCMCSampler(
    network=scenario.network,
    cost_fn=cost_fn,
    proposals=proposals,
    proposal_weights=weights,
    beta=BETA,
    mode='optimization',
    rng=rng,
)
sampler.initialize(Schedule.random(scenario.network, rng))
result = sampler.run(n_steps=N_STEPS_QUICK, log_every=5000)

print(f'=== SZYBKA OPTYMALIZACJA ===')
print(f'  Acceptance rate: {result.acceptance_rate:.3f}  (cel: 0.20–0.40)')
print(f'  Best cost: {result.best_cost:.2f}')
print(f'  Mean cost (ostatnie próbki): {np.mean(result.costs):.2f}' if result.costs else '')

if scenario.metadata.optimal_solution is not None:
    opt_sched = Schedule(offsets=scenario.metadata.optimal_solution, period=scenario.network.period)
    opt_cost = cost_fn(opt_sched, scenario.network)
    gap = (result.best_cost - opt_cost) / opt_cost if opt_cost > 1e-9 else result.best_cost - opt_cost
    print(f'  Optimal cost: {opt_cost:.2f}')
    print(f'  Gap to optimum: {gap:.2%}')

# Ostrzeżenia
if result.acceptance_rate < 0.05:
    print('  UWAGA: acceptance < 5% → beta za duże lub sigma za duże')
elif result.acceptance_rate > 0.80:
    print('  UWAGA: acceptance > 80% → beta za małe, chain błądzi')

---
## 5. Parallel Tempering — framework kalibracji

### Kiedy używać PT

PT pomaga gdy single-chain utyka w lokalnych minimach. Diagnoza:
- **multi_hub_mesh**: R-hat = 10 492 dla single-chain (nie zbiega) → PT konieczne
- **hub/chain**: single-chain zbiega w < 50k kroków → PT zbędne

### Mechanizm PT w tym projekcie

```
M replik z beta: beta_0 < beta_1 < … < beta_{M-1}
  beta_0 = beta_min (najgorętsza, idx=0)   — swobodna eksploracja
  beta_{M-1} = beta_max (najzimniejsza)    — target distribution

Co swap_interval kroków: próba zamiany między wszystkimi parami (i, i+1)
  swap_alpha = exp((beta_{i+1} - beta_i) * (cost_i - cost_{i+1}))
  accept z prob min(1, swap_alpha)

Inercja: zimne repliki preferują drogi IC, gorące — uniform
```

### Problem 1: Swap rates < 1% — za duże Δβ

**Przyczyna:** Koszty między replikami różnią się za bardzo.  
`(beta_{i+1} - beta_i) × (cost_i - cost_{i+1}) >> 1` → swap_prob ≈ 0

**Diagnoza:** `probe_betas` pokazuje, od jakiego beta chain zaczyna się mieszać.

**Rozwiązanie A** (proste): zawęź zakres beta
```
Stare: beta_min=0.1, beta_max=5.0, ratio=50×  → swap < 1%
Nowe:  beta_min=0.5, beta_max=3.0, ratio=6×   → swap 20–40%
```

**Rozwiązanie B** (auto-kalibracja, patrz komórka poniżej)

### Problem 2: Zamrożony chain — beta_max w złym zakresie

Jeśli acceptance rate jest identyczny dla beta=0.5 i beta=3.0 (jak dla random_network_hard),  
PT **nie ma fizycznego sensu** przy tych beta — repliki są wszystkie w tym samym stanie.

**Sygnatura problemu z pt_calibration.json:**  
- calibrated Δβ ≈ 0.000135 → beta range [2.9991, 3.0000]  
- Swap rates zdrowe (0.3–0.5), ale PT = parallel hill climbing, nie PT

**Co wtedy:**
1. Ustaw `beta_max` w zakresie gdzie chain **naprawdę** się miesza (z Kroku 2)
2. Dla random_network_hard: `beta_max ≈ 0.001`, nie 3.0
3. Alternatywa: użyj multi-start (M niezależnych łańcuchów bez swap)

### Wyniki empiryczne

#### multi_hub_mesh_hard (18L, 25T, 500k kroków)

| Config | Best cost | Swap avg | Swap min |
|--------|-----------|----------|----------|
| **Baseline (1 chain)** | **33 145** | — | — |
| M=8 (beta 0.5–3.0) | 4 715 | 0.44 | 0.001 |
| M=12 (beta 0.5–3.0) | **4 678** | 0.53 | 0.014 |
| M=16+inertia (beta 0.5–3.0) | **4 678** | 0.54 | 0.004 |
| M=8 (kalibrowany beta 1.9–3.0) | **4 678** | 0.45 | 0.004 |

**PT daje 7× lepszy koszt** niż single-chain na tym scenariuszu.

#### random_network_hard (20L, 82T, 500k kroków)

| Config | Best cost | Faktyczny mechanizm |
|--------|-----------|---------------------|
| Baseline (1 chain) | 96 535 | hill climbing |
| PT M=8 (beta 0.5–3.0) | 92 196 | parallel hill climbing |
| PT M=8 kalibrowany (beta 2.999–3.0) | **86 163** | par. hill climbing + drobny gradient |

Poprawa 11% — z parallel restartów, nie z mechanizmu PT.

In [ ]:
# KROK 6 (PT): Auto-kalibracja beta i uruchomienie PT
#
# Kalibracja opiera się na aproksymacji Gaussowskiej swap rates:
#   swap_rate ≈ erfc(Δβ · σ_E / √2) / 2
#   => Δβ = √2 · erfinv(1 - 2·target_swap) / σ_E
# gdzie σ_E = std kosztów przy beta_max (krótki run).

# CONFIG
BETA_MAX      = 3.0    # zimny łańcuch — ustaw tak żeby acceptance ≈ 20–40%
N_REPLICAS    = 8      # rekomendowane: 8–12 dla sieci medium
TARGET_SWAP   = 0.23   # docelowy swap acceptance rate
N_STEPS_PT    = 500_000  # całkowity budżet (dzielony przez N_REPLICAS)
USE_INERTIA   = True   # True dla sieci multi-modalnych (IC/Regional/Local)

# Krok A: Sprawdź sigma przy beta_max
print(f'Szacowanie sigma przy beta_max={BETA_MAX}...')
stats = estimate_cost_stats(scenario, BETA_MAX, n_warmup=3000, n_sample=8000, seed=SEED)
print(f'  mean={stats.mean:.1f}  std={stats.std:.1f}  accept={stats.acceptance_rate:.3f}')

# Diagnoza: czy beta_max jest sensowne?
if stats.acceptance_rate < 0.05:
    print('OSTRZEZENIE: acceptance < 5% przy beta_max — rozważ mniejsze beta_max')
elif stats.std < 1.0:
    print('OSTRZEZENIE: std bardzo male — chain juz znalazl minimum, PT nie pomoze')
    print('  Kalibrator moze wygenerowac bardzo waski zakres beta (quasi-identyczne repliki)')

# Krok B: Kalibracja
print(f'\nKalibracja beta schedule (M={N_REPLICAS}, target_swap={TARGET_SWAP})...')
betas = calibrate_beta_schedule(
    scenario, BETA_MAX, n_replicas=N_REPLICAS,
    target_swap=TARGET_SWAP, n_warmup=3000, n_sample=8000, seed=SEED,
)
print(f'  beta_min={betas[0]:.5f}, beta_max={betas[-1]:.3f}')
print(f'  delta_beta={betas[1]-betas[0]:.6f}')

# Krok C: Uruchom PT z kalibrowanym harmonogramem
print(f'\nUruchamianie PT (M={N_REPLICAS}, budget={N_STEPS_PT})...')
pt_result = run_pt_with_betas(
    scenario, betas,
    n_steps_per_replica=N_STEPS_PT // N_REPLICAS,
    use_inertia=USE_INERTIA,
    seed=SEED,
    checkpoint_every=20000,
)

print(f'\n=== WYNIKI PT ===')
print(f'  Best cost: {pt_result["best_cost"]:.2f}')
print(f'  Swap avg:  {pt_result["swap_avg"]:.3f}  (cel: 0.20–0.40)')
print(f'  Swap min:  {pt_result["swap_min"]:.3f}  (< 0.05 = problem z para 0-1)')
print(f'  Accept per replica: {[f"{r:.2f}" for r in pt_result["per_replica_acceptance"]]}')

# Ostrzeżenia:
if pt_result['swap_avg'] < 0.05:
    print('OSTRZEZENIE: swap_avg < 5% — repliki nie komunikuja sie. Zawez zakres beta.')
if pt_result['swap_avg'] > 0.50:
    print('INFO: swap_avg > 50% — repliki bardzo podobne. Mozna rozszerzyc zakres.')
if pt_result['swap_min'] < 0.01:
    print('OSTRZEZENIE: swap_min < 1% — waskogardlo w parze 0-1 (najzimniejsza).')
    print('  Dodaj wiecej replik w zimnym zakresie lub zwieksz beta_max.')

In [ ]:
# PT przez standardowy interface (bez auto-kalibracji)
# Używaj gdy znasz już dobry zakres beta z poprzednich eksperymentów.

# CONFIG
N_REPLICAS_STD = 8
BETA_MIN_STD   = 0.5
BETA_MAX_STD   = 3.0
SWAP_INTERVAL  = 10
N_STEPS_STD    = 200_000

rng2 = np.random.default_rng(SEED)
cost_fn2 = build_cost_fn(scenario)
proposals2, weights2 = build_proposals(scenario, rng2)

inertia = InertiaConfig(modality_inertia={'IC': 20.0, 'Regional': 5.0, 'bus': 1.0})

pt = ParallelTempering(
    network=scenario.network,
    cost_fn=cost_fn2,
    proposals=proposals2,
    proposal_weights=weights2,
    n_replicas=N_REPLICAS_STD,
    beta_min=BETA_MIN_STD,
    beta_max=BETA_MAX_STD,
    swap_interval=SWAP_INTERVAL,
    mode='optimization',
    inertia_config=inertia,
    rng=rng2,
)
pt.initialize()

print(f'Temperature schedule: {[f"{b:.3f}" for b in pt.temperature_schedule]}')

res = pt.run(n_steps=N_STEPS_STD // N_REPLICAS_STD, log_every=0)

print(f'Best cost: {res.best_cost:.2f}')
print(f'Swap rates: {[f"{r:.3f}" for r in res.swap_acceptance_rates]}')
print(f'Per-replica acceptance: {[f"{r:.3f}" for r in res.per_replica_acceptance_rates]}')

---
## 6. Diagnostyki zbieżności

### Interpretacja R-hat i ESS

| Metryka | Co mierzy | Kryterium | Problem gdy... |
|---|---|---|---|
| R-hat | Spójność między łańcuchami | < 1.1 | R-hat > 1.5 → łańcuchy eksplorują różne regiony |
| ESS/chain | Efektywna liczba niezależnych próbek | > 50 | ESS < 10 → bardzo wysoka autokorelacja |
| ESS/step | Efektywność per krok | > 0.01 | < 0.001 → rozważ inne propozycje |
| acceptance | Frakcja zaakceptowanych | 20–40% | < 5% lub > 80% → źle dobrane parametry |

**Wyniki z diagnostics_full.json (50k kroków, 4 łańcuchy):**

| Scenariusz | R-hat | ESS | Kroki do zbieżności |
|---|---|---|---|
| hub_easy | **1.003** | 700 | 16 000 |
| hub_hard | **1.005** | 326 | 42 000 |
| chain_medium | **1.001** | 1 911 | 10 000 |
| conflicting_easy | **1.001** | 2 600 | 6 000 |
| **multi_hub_mesh_easy** | **10 492** | 570 | **>50 000** (nie zbiegł!) |

In [ ]:
# KROK 4: Diagnostyka zbieżności (multi-chain)

# CONFIG
N_CHAINS_CONV   = 4
N_STEPS_CONV    = 30000   # per chain
CHECKPOINT_CONV = 2000
BETA_CONV       = 1.0

print(f'Uruchamianie {N_CHAINS_CONV} lancuchow x {N_STEPS_CONV} krokow...')
conv_result = run_convergence_diagnostic(
    scenario,
    n_chains=N_CHAINS_CONV,
    n_steps=N_STEPS_CONV,
    checkpoint_every=CHECKPOINT_CONV,
    beta=BETA_CONV,
    seed=SEED,
)

print(f'\nWyniki:')
print(f'  R-hat finalny: {conv_result.final_r_hat:.4f}  (cel: < 1.1)')
print(f'  ESS total:     {conv_result.final_ess_total:.0f}')
print(f'  Zbieznosc:     {conv_result.converged}')
print(f'  Kroki do zbieznosci: {conv_result.steps_to_convergence}')
print(f'  Czas: {conv_result.wall_time_s:.1f}s')

# Wykres R-hat w czasie
if conv_result.checkpoints:
    steps = [cp.step for cp in conv_result.checkpoints]
    rhats = [cp.r_hat for cp in conv_result.checkpoints]
    ess   = [cp.ess_total for cp in conv_result.checkpoints]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.plot(steps, rhats, 'o-', markersize=4)
    ax1.axhline(1.1, color='red', linestyle='--', alpha=0.6, label='threshold 1.1')
    ax1.set_xlabel('Steps'); ax1.set_ylabel('R-hat'); ax1.legend()
    ax1.set_title('Gelman-Rubin R-hat')
    
    ax2.plot(steps, ess, 'o-', markersize=4, color='steelblue')
    ax2.axhline(100 * N_CHAINS_CONV, color='red', linestyle='--', alpha=0.6, label='min ESS')
    ax2.set_xlabel('Steps'); ax2.set_ylabel('Total ESS'); ax2.legend()
    ax2.set_title('Effective Sample Size')
    plt.tight_layout()
    plt.show()

In [ ]:
# Ręczna diagnostyka na własnych łańcuchach
# Użyj gdy masz już uruchomione łańcuchy i chcesz sprawdzić zbieżność.

# Przykład: 3 łańcuchy, każdy 5000 próbek
# chain1_costs = [...]   # lista floatów
# chain2_costs = [...]   # ...
# chain3_costs = [...]   # ...

# diag = MCMCDiagnostics([chain1_costs, chain2_costs, chain3_costs])
# print(diag.summary())
# print(f'R-hat: {diag.r_hat():.4f}')
# print(f'ESS:   {diag.ess_total():.0f}')
# print(f'Zbiezny: {diag.is_converged(r_hat_threshold=1.1, min_ess_threshold=100)}')

# Przykład na danych syntetycznych:
dummy_chains = [
    np.random.default_rng(i).normal(loc=5.0, scale=0.5, size=2000).tolist()
    for i in range(3)
]
diag = MCMCDiagnostics(dummy_chains)
print('Przykladowa diagnostyka (3 syntetyczne lancuchy):')
print(diag.summary())

---
## 7. Porównanie propozycji ruchów

### Kiedy używać której propozycji

| Propozycja | Kiedy | Unikaj gdy |
|---|---|---|
| `SingleLineShift` | Zawsze — bazowa | — |
| `UniformLineShift` | Uzupełnienie SLS, eksploracja jednorodna | — |
| `ComponentMove` | Wiele przesiadek (>5), linie powiązane | Brak przesiadek |
| `SmartRepair` | Wiele naruszeń w_min, chcesz max ESS/step | Brak przesiadek |

**Uwaga SmartRepair:** ma najniższy acceptance (7.4%), ale najwyższy ESS/step (6× lepszy od full_mix).  
Oznacza to: rzadko akceptuje, ale każda akceptacja to duży skok w lepsze miejsce.

In [ ]:
# KROK 5: Porównanie propozycji

# CONFIG
N_STEPS_PROP = 10000
N_CHAINS_PROP = 2
BETA_PROP = 1.0

print(f'Porownanie propozycji ({N_STEPS_PROP} krokow x {N_CHAINS_PROP} lancuchow)...')
prop_result = run_proposal_comparison(
    scenario,
    n_steps=N_STEPS_PROP,
    n_chains=N_CHAINS_PROP,
    beta=BETA_PROP,
    seed=SEED,
)

print(f'\n{"Propozycja":<25} {"Accept":>8} {"Best":>12} {"ESS/step":>10} {"ms/step":>9}')
print('-' * 70)
for e in prop_result.entries:
    print(f'{e.proposal_name:<25} {e.acceptance_rate:>8.3f} {e.best_cost:>12.1f} '
          f'{e.ess_per_step:>10.4f} {e.time_per_step_ms:>9.3f}')

# Rekomendacja
best_ess = max(prop_result.entries, key=lambda e: e.ess_per_step)
best_cost = min(prop_result.entries, key=lambda e: e.best_cost)
print(f'\nNajlepsza ESS/step: {best_ess.proposal_name} ({best_ess.ess_per_step:.4f})')
print(f'Najlepszy best_cost: {best_cost.proposal_name} ({best_cost.best_cost:.1f})')

---
## 8. Kalibracja funkcji kosztu

### 8.1 Wrażliwość na barierę M

**Cel:** znaleźć M takie, że:
- Naruszenia (w < w_min) są silnie karane
- Ale chain może opuścić stany naruszające (M nie może być ∞)
- Praktyczna zasada: `M = 1e5`–`1e6` dla przesiadek, `1e7` dla obrotów taboru

**Wyniki empiryczne (hard instancje, 5000 kroków):**

| Scenariusz | M=1e3 | M=1e4 | M=1e5–1e7 | violation % |
|---|---|---|---|---|
| multi_hub_mesh_hard (18L, 25T) | 4739 | **4703** | **4703** | ~12% (stałe) |
| random_network_hard (20L, 82T) | 175 342 | **106 383** | **106 383** | **100% (stałe!)** |

**Kluczowe wnioski:**
- Od M ≥ 1e4 wyniki są **identyczne** — bariera jest już wystarczająco stroma.
  Zwiększanie M powyżej 1e5 nic nie zmienia (ani violation rate, ani koszt).
- **M=1e5 jako domyślne** — bezpieczny margines ponad minimum.
- Violation rate jest **cechą struktury sieci**, nie doboru M:
  - multi_hub_mesh_hard: 12% naruszeń przy *każdym* M → sieć ma ciasne ograniczenia
  - random_network_hard: 100% naruszeń przy *każdym* M → sprzeczne ograniczenia niemożliwe do jednoczesnego spełnienia (gęstość transferów 0.7 przy 20 liniach)
  - Miękka bariera działa poprawnie: minimalizuje naruszenia, ale zera nie osiągnie dla niespójnych sieci.

### 8.2 Wagi λ (pax, ops, rob)

**Trade-off:** zwiększenie `ops_weight` → lepsze obroty taboru, kosztem jakości przesiadek.  
Analiza Pareto pozwala znaleźć kompromis bez arbitralnego wyboru wag.

**Wyniki lambda sweep (hard instancje, 9 konfiguracji, 5000 kroków):**

| lambda_rob | multi_hub PAX | multi_hub ROB | random_net PAX | random_net ROB |
|---|---|---|---|---|
| **0.01** | **4703** | 6.0 | **106 383** | 3072 |
| **0.1** | **4703** | 6.0 | **106 383** | 3072 |
| **1.0** | 33 259 (+607%) | 63.5 | 108 674 (+2%) | 2922 |

**Kluczowe wnioski:**
- `lambda_rob=1.0` powoduje **drastyczne pogorszenie PAX**: 7× na multi_hub_mesh, 2% na random_net.
  Zysk ROB jest nieproporcjonalny (10× wzrost wartości bezwzględnej ROB na multi_hub przy 607% pogorszeniu PAX).
- `lambda_ops` nie ma żadnego wpływu, gdy scenariusz nie posiada turnaroundów (`cost_ops=0.0` dla wszystkich konfiguracji).
- **Rekomendacja: `lambda_rob ≤ 0.1`** — wyższe wartości niszczą jakość przesiadek bez proporcjonalnego zysku robustności.
- Analiza Pareto PAX vs OPS jest zdegenerowana (jeden punkt) — sensowna dopiero dla scenariuszy `full_cost` z aktywnymi turnaroundami.

**Domyślne wagi** (z diagnostic_runner.py, empirycznie potwierdzone):
```python
pax_weight = 1.0   # pasażerowie (przesiadki) — skala referencyjna
ops_weight = 0.1   # operacyjne (obroty) — bez efektu gdy brak turnaroundów
rob_weight = 0.05  # robustność — proxy, nie formalna minimalizacja; NIE dawaj > 0.1
```

In [ ]:
# Wrażliwość na M (barierę)

# CONFIG
M_VALUES_TEST = [1e3, 1e4, 1e5, 1e6, 1e7]
N_STEPS_BARRIER = 3000

barrier_result = run_barrier_sensitivity(
    scenario,
    M_values=M_VALUES_TEST,
    n_steps=N_STEPS_BARRIER,
    beta=1.0,
    seed=SEED,
)

print(f'{"M":>10} {"Best Cost":>12} {"Violations%":>12} {"Accept":>8}')
print('-' * 50)
for p in barrier_result.points:
    print(f'{p.M_barrier:>10.0e} {p.best_cost:>12.2f} {p.violation_rate*100:>11.1f}% {p.acceptance_rate:>8.3f}')

# Interpretacja:
# M=1e3: violations duże → bariera za mała, chain ignoruje naruszenia
# M=1e6: violations ≈ 0 → dobra bariera
# M=1e7: violations = 0, ale best_cost może rosnąć (chain nie eksploruje)

In [ ]:
# Sweep po wagach λ — analiza trade-offów
# UWAGA: działa tylko dla scenariuszy z turnaroundami (full_cost, multi_hub_mesh)

# CONFIG
LAMBDA_GRID = {
    'pax': [1.0],
    'ops': [0.01, 0.1, 1.0, 5.0],
    'rob': [0.01, 0.1, 0.5],
}
N_STEPS_LAMBDA = 3000

# Użyj scenariusza z turnaroundami
scenario_full = factory.create('full_cost', difficulty='easy')

lambda_result = run_lambda_sweep(
    scenario_full,
    lambda_grid=LAMBDA_GRID,
    n_steps=N_STEPS_LAMBDA,
    beta=1.0,
    seed=SEED,
)

print(f'{"pax_w":>6} {"ops_w":>6} {"rob_w":>6} {"Best":>10} {"PAX":>10} {"OPS":>10} {"ROB":>10}')
print('-' * 65)
for p in lambda_result.points[:10]:  # pierwsze 10
    print(f'{p.lambda_pax:>6.2f} {p.lambda_ops:>6.2f} {p.lambda_rob:>6.2f} '
          f'{p.best_cost:>10.2f} {p.cost_pax:>10.2f} {p.cost_ops:>10.2f} {p.cost_rob:>10.2f}')

# Pareto front PAX vs OPS
pareto = compute_pareto_front(lambda_result.points, 'cost_pax', 'cost_ops')
print(f'\nPareto front (PAX vs OPS): {len(pareto.points)} punktow Pareto-optymalnych')

---
## 9. Benchmark metod

Porównanie MCMC, Simulated Annealing, Random Search i PT na tym samym budżecie obliczeniowym.

**Wyniki z notebook 06 (BUDGET=5000, MODE=quick):**

| Scenariusz | Random | MCMC | SA | PT (M=4) |
|---|---|---|---|---|
| hub_easy | 78.4 | 3.1 | 3.1 | 3.1 |
| chain_easy | 0.7 | 0.7 | 0.7 | 0.7 |
| multi_hub_mesh_easy | 23 173 | 8 961 | **3 114** | 3 141 |

SA wygrywa na multi_hub_mesh przy małym budżecie — szybsze schladzanie niż MCMC przy stałym beta.

In [ ]:
# Benchmark metod na wybranym scenariuszu

# CONFIG
METHODS = ['random', 'mcmc', 'sa', 'pt']
BUDGET  = 10000   # ewaluacje funkcji kosztu per metoda

bench_result = run_benchmark_comparison(
    scenario,
    methods=METHODS,
    budget=BUDGET,
    seed=SEED,
)

print(f'=== BENCHMARK: {bench_result.scenario_name} ===')
print(f'{"Metoda":<30} {"Best Cost":>12} {"Czas (s)":>10} {"Ewaluacje":>12}')
print('-' * 70)
for method, cost, t, n_eval in zip(
        bench_result.methods, bench_result.best_costs,
        bench_result.wall_times, bench_result.n_evaluations):
    print(f'{method:<30} {cost:>12.2f} {t:>10.2f} {n_eval:>12}')

# Simulated Annealing oddzielnie (więcej kontroli)
print('\n--- Simulated Annealing osobno ---')
sa_result = run_simulated_annealing(
    scenario,
    n_steps=BUDGET,
    T_init=100.0,
    T_final=0.01,
    cooling_schedule='geometric',
    seed=SEED,
)
print(f'  Best: {sa_result.best_cost:.2f}')
print(f'  Final temp: {sa_result.final_temperature:.4f}')
print(f'  Accept rate: {sa_result.acceptance_rate:.3f}')

---
## 10. Skalowanie i profilowanie

### Dominujący bottleneck zależy od |transfers|

Punkt przegięcia leży w okolicy **|T| ~ 20–30 transferów**:
- **Poniżej**: Proposal dominuje (losowanie linii, SmartRepair, RNG)
- **Powyżej**: Cost Evaluation dominuje (liniowe skalowanie z |T|)

| Scenariusz | |L| | |T| | Proposal % | Cost % | ms/krok |
|---|---|---|---|---|---|
| hub_easy | 6 | 9 | **75%** | 21% | 0.175 |
| multi_hub_mesh_easy | 12 | 16 | **57%** | 40% | 0.142 |
| multi_hub_mesh_medium | 18 | 25 | **52%** | 45% | 0.208 |
| multi_hub_mesh_hard | 18 | 25 | **43%** | **55%** | 0.125 |
| random_network_medium | 12 | 38 | 35% | **63%** | 0.172 |
| **random_network_hard** | **20** | **82** | 22% | **77%** | **0.265** |

**Uwaga:** multi_hub_mesh_hard i medium mają tę samą liczbę linii/transferów (18L, 25T), ale różny czas per krok — wynika to z różnic w strukturze sieci i losowania propozycji.

### Skalowanie z rozmiarem sieci — liniowe (R²=0.92)

```
t ≈ 0.0032 × |T| + 0.056 ms
```

- **0.056 ms** — stały narzut (propozycja, acceptance, RNG)
- **0.0032 ms/transfer** — koszt marginalny per transfer

Weryfikacja dla random_network_hard (82T):  
przewidywane = 0.056 + 0.0032 × 82 = **0.318 ms**, zmierzone = **0.339 ms** ✓

| Sieć | |L| | |T| | ms/step | steps/s |
|---|---|---|---|---|
| chain_easy | 3 | 2 | 0.06 | 15 982 |
| hub_easy | 6 | 9 | 0.14 | 7 257 |
| multi_hub_mesh_easy | 12 | 16 | 0.12 | 8 204 |
| hub_hard | 12 | 36 | 0.16 | 6 291 |
| random_network_medium | 12 | 38 | 0.16 | 6 264 |
| **random_network_hard** | **20** | **82** | **0.34** | **2 947** |

### Koszt per transfer maleje z rozmiarem (efekt amortyzacji)

| Scenariusz | |T| | ms/transfer |
|---|---|---|
| hub_easy | 9 | 0.015 |
| multi_hub_mesh_hard | 25 | 0.005 |
| random_network_hard | 82 | **0.003** |

Sam koszt transferu jest O(1), ale udział stałego narzutu (propozycja, acceptance) maleje z rozmiarem sieci.

### Rekomendacje optymalizacyjne

| Gdy | Bottleneck | Akcja |
|---|---|---|
| |T| < 20 | Proposal (>50% czasu) | Optymalizuj dobór propozycji; Numba/Cython dla SmartRepair |
| |T| ≥ 50 | Cost Evaluation (>70%) | **Inkrementalny delta_cost()** — przeliczaj tylko transfery dotknięte zmienionymi liniami |
| |T| ≥ 100 | Cost Evaluation (>85%) | Cache wyników per linia + inkrementalny koszt |

**Dla random_network_hard (82T)**: Cost Evaluation = 77% → priorytet to `delta_cost()`.  
Przy 82 transferach pełne przeliczenie kosztuje 0.203 ms/krok. Inkrementalny koszt (tylko zmienione linie) skróciłby to ~5–10×.

In [ ]:
# Profilowanie — ile czasu zajmuje każdy komponent

N_STEPS_PROF = 1000

prof_result = run_profiling_analysis(scenario, n_steps=N_STEPS_PROF, seed=SEED)

print(f'=== PROFILING: {prof_result.scenario_name} ===')
print(f'Total: {prof_result.total_time_s*1000:.1f} ms / {prof_result.n_steps} steps')
print(f'{"Komponent":<25} {"ms/step":>10} {"%":>8}')
print('-' * 45)
for p in prof_result.breakdown:
    print(f'{p.component:<25} {p.time_ms:>10.3f} {p.percentage:>8.1f}%')

# Skalowanie
print('\nSkalowanie (2000 krokow per scenariusz)...')
scaling = run_scaling_analysis(n_steps=500, beta=1.0, seed=SEED)  # 500 dla szybkości
print(f'{"Scenariusz":<35} {"L":>4} {"T":>4} {"ms/step":>9} {"steps/s":>9}')
print('-' * 65)
for p in scaling.points:
    sps = 1000 / p.time_per_step_ms if p.time_per_step_ms > 0 else 0
    print(f'{p.scenario_name:<35} {p.n_lines:>4} {p.n_transfers:>4} '
          f'{p.time_per_step_ms:>9.3f} {sps:>9.0f}')

---
## 11. Zapis i wczytywanie wyników

In [ ]:
# Zapis wyników do JSON
import json
from dataclasses import asdict

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'my_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Przykład zapisu własnych wyników
my_results = {
    'scenario': scenario.metadata.name,
    'n_lines': scenario.metadata.n_lines,
    'n_transfers': scenario.metadata.n_transfers,
    'landscape': {
        'min': landscape.min_cost,
        'median': landscape.median,
        'mean': landscape.mean,
        'std': landscape.std,
    },
    # Dodaj tu inne wyniki (conv_result, prop_result, pt_result, ...)
}

out_path = OUTPUT_DIR / f'{scenario.metadata.name}_analysis.json'
with open(out_path, 'w') as f:
    json.dump(my_results, f, indent=2)
print(f'Zapisano do {out_path}')

# Wczytanie
with open(out_path) as f:
    loaded = json.load(f)
print(f'Wczytano: {loaded["scenario"]}, min_cost={loaded["landscape"]["min"]:.2f}')

# Wczytanie istniejących wyników diagnostycznych
diag_path = PROJECT_ROOT / 'results' / 'diagnostics' / 'diagnostics_full.json'
if diag_path.exists():
    with open(diag_path) as f:
        diag_data = json.load(f)
    print(f'\nWczytano diagnostics_full.json:')
    print(f'  Mode: {diag_data["mode"]}, Seed: {diag_data["seed"]}')
    for name, r in diag_data['convergence'].items():
        status = 'OK' if r['converged'] else 'FAIL'
        print(f'  [{status}] {name}: R-hat={r["r_hat"]:.3f}, ESS={r["ess_total"]:.0f}')

In [ ]:
# Uruchomienie pełnego benchmark suite (wiele scenariuszy naraz)
# Używa configs/synthetic_benchmark.yaml

# Z linii komend:
# python -m src.experiments.benchmark.run_synthetic_benchmark \
#     --config configs/synthetic_benchmark.yaml \
#     --output results/my_benchmark \
#     --seed 42

# Lub programowo:
from src.experiments.benchmark.run_synthetic_benchmark import (
    SyntheticBenchmarkConfig, run_synthetic_benchmark_suite, save_results
)

bench_config = SyntheticBenchmarkConfig(
    scenarios=[
        {'type': 'hub', 'difficulty': 'easy'},
        {'type': 'chain', 'difficulty': 'easy'},
        {'type': 'multi_hub_mesh', 'difficulty': 'easy'},
    ],
    n_chains=2,
    n_steps_per_chain=5000,   # mało dla przykładu
    beta=1.0,
    pax_weight=1.0,
    ops_weight=0.1,
    rob_weight=0.05,
)

print('Uruchamianie benchmark suite...')
suite = run_synthetic_benchmark_suite(bench_config, base_seed=SEED)

print(f'\n{"Scenariusz":<30} {"Best":>12} {"Gap":>8} {"R-hat":>8} {"Czas":>8}')
print('-' * 75)
for r in suite.results:
    gap_str = f'{r.gap_to_optimum:.2%}' if r.gap_to_optimum is not None else 'N/A'
    rhat_str = f'{r.diagnostics["r_hat"]:.3f}' if r.diagnostics.get('r_hat') else 'N/A'
    print(f'{r.scenario_name:<30} {r.best_cost:>12.2f} {gap_str:>8} {rhat_str:>8} {r.runtime_seconds:>7.1f}s')

# Zapis
save_results(suite, OUTPUT_DIR)
print(f'\nZapisano do {OUTPUT_DIR}')

---
## Szybka lista kontrolna dla nowej sieci

```
[ ] 1. Landscape analysis
      → std/mean > 1? Wielu lokalnych minimów — rozważ PT
      → min ≈ M (barrier)? Problem z M — zmniejsz lub sprawdź dane

[ ] 2. Probe betas (0.0001 do 3.0)
      → acceptance stały dla wielu beta? Chain zamrożony — użyj niższego beta_max
      → Znajdź zakres gdzie acceptance = 20–40%

[ ] 3. Szybka optymalizacja (10–20k kroków)
      → acceptance < 5%? Zmniejsz beta lub sigma
      → acceptance > 80%? Zwiększ beta
      → Czy best_cost poprawia się monotonicz? Jeśli nie — chain utknął

[ ] 4. Zbieżność (4 łańcuchy, 30–50k kroków)
      → R-hat < 1.1? Zbiegło ✓
      → R-hat > 5? Potrzebny PT lub więcej kroków
      → ESS > 100 per chain? ✓

[ ] 5. Porównanie propozycji
      → Wybierz mix o najlepszym ESS/step
      → Jeśli wiele naruszeń → zwiększ smart_repair weight

[ ] 6. PT (jeśli chain nie zbiega lub utyka)
      a. estimate_cost_stats(beta_max) → std?
      b. Jeśli std > 0 i acceptance sensowne:
         calibrate_beta_schedule() → betas
         run_pt_with_betas() → sprawdź swap_avg, swap_min
      c. swap_avg < 5%? → beta_max za duże lub za mało replik
      d. swap_min < 1%? → wąskie gardło przy parze 0-1, dodaj repliki w zimnym zakresie
      e. Acceptance ≈ stały dla wielu beta? → chain zamrożony przy wszystkich tych beta!
         Użyj beta_max odpowiednio mniejsze.

[ ] 7. Kalibracja kosztu (opcjonalnie)
      → barrier sensitivity: znajdź M gdzie violations → 0
      → lambda sweep: sprawdź trade-off PAX vs OPS

[ ] 8. Benchmark
      → Porównaj MCMC vs SA vs PT przy tym samym budżecie
      → SA bywa lepszy przy małym budżecie (szybkie schladzanie)

[ ] 9. Zapis wyników do JSON
```

---
### Znane pułapki

1. **Arytmetyka modularna**: offsety są mod `T_PERIOD=1440`. Czas oczekiwania `w = (-delta_phase) % frequency` — zawsze ≥ 0.

2. **Bariery w krajobrazie**: jeśli landscape.mean ≈ M×n_transfers → większość losowych rozkładów narusza constrainty. Nie panikuj — MCMC poradzi sobie, ale potrzebuje więcej kroków na wyjście z obszarów naruszeń.

3. **PT dla dużych sieci (>50 przesiadek)**: swap probability `exp(-Δβ × ΔE)`. Przy kosztach ~100k, nawet małe Δβ=0.001 daje `exp(-0.001 × 10000) = exp(-10) ≈ 0`. Auto-kalibracja może wyprodukować Δβ~0.0001 — bardzo wąski zakres, de facto parallel restarts.

4. **Inercja PT**: działa dla sieci z rzeczywistą hierarchią modalną (IC > Regional > Local). Dla `random_network` (losowe modality) inercja **może szkodzić** — wyłącz `use_inertia=False`.

5. **SmartRepair i Hastings**: używa dyskretnych mas prawdopodobieństwa (nie gęstości), bo delta jest zaokrąglana do int. To poprawne — nie zmieniaj na ciągłe gęstości.

6. **Mode A vs Mode B**: `mode='sampling'` tworzy ProposalAdapter z diminishing adaptation (gamma → 0). `mode='optimization'` śledzi best-so-far bez formalnych gwarancji stacjonarności. **Używaj Mode B do optymalizacji.**